In [1]:
import pandas as pd
messages = pd.read_csv('./Spam.csv', sep = ',', names = ['label', 'message'], skiprows = 1)
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
#optional
import pandas as pd
messages = pd.read_csv('./Spam.csv', sep = ',')
messages.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
messages.describe()

,Category,Message
count,5572,5572
unique,2,5157
top,ham,"Sorry, I'll call later"
freq,4825,30


In [5]:
messages.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
import pandas as pd
messages = pd.read_csv('./Spam.csv', sep = ',', names = ['label', 'message'], skiprows = 1)
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
messages['lenght']=messages['message'].apply(len)

In [9]:
messages.head()

,label,message,lenght
0,ham,"Go until jurong point, crazy.. Available only ...",111
1,ham,Ok lar... Joking wif u oni...,29
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
3,ham,U dun say so early hor... U c already then say...,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",61


In [11]:
#NLP - Stopwords removing - Text pre-processing 
from nltk.corpus import stopwords
import nltk
import string
nltk.download('stopwords', quiet=True)

def text_process(mess):
    nopunc = [char for char in mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]
def text_process(mess):
    nopunc = [char for char in mess if char not in string.punctuation] 
    nopunc = ''.join(nopunc) 
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

In [12]:
messages['message'].head().apply(text_process)

0    [Go, jurong, point, crazy, Available, bugis, n...
1                       [Ok, lar, Joking, wif, u, oni]
2    [Free, entry, 2, wkly, comp, win, FA, Cup, fin...
3        [U, dun, say, early, hor, U, c, already, say]
4    [Nah, dont, think, goes, usf, lives, around, t...
Name: message, dtype: object

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
bow_transformer = CountVectorizer(analyzer=text_process).fit(messages['message'])

In [15]:
print(len(bow_transformer.vocabulary_))

11422


In [22]:
messages_bow = bow_transformer.transform(messages['message'])
print('Shape of Sparse Matrix: ', messages_bow.shape)
print('Amount of Non-Zero occurences: ', messages_bow.nnz)
print('Sparsity: %.2f%%' % (100.0 * messages_bow.nnz / (messages_bow.shape[0] * messages_bow.shape[1])))

Shape of Sparse Matrix:  (5572, 11422)
Amount of Non-Zero occurences:  50500
Sparsity: 0.08%


In [29]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer().fit(messages_bow)
tfidf4 = tfidf_transformer.transform(messages_bow)
print(tfidf4.toarray())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [31]:
messages_tfidf = tfidf_transformer.transform(messages_bow) 
print(messages_tfidf.shape)
from sklearn.naive_bayes import MultinomialNB  
spam_detect_model = MultinomialNB() # model creation 
spam_detect_model.fit(messages_tfidf, messages['label']) # training 
from sklearn.metrics import classification_report  # model evl.  
all_predictions =  spam_detect_model.predict(messages_tfidf) 
print(classification_report(messages['label'], all_predictions))
print(all_predictions) # model testing

(5572, 11422)
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      4825
        spam       1.00      0.85      0.92       747

    accuracy                           0.98      5572
   macro avg       0.99      0.92      0.95      5572
weighted avg       0.98      0.98      0.98      5572

['ham' 'ham' 'spam' ... 'ham' 'ham' 'ham']


In [ ]:
msg4=messages['message'][3]
bow4 = bow_transformer.transform([msg4])
tfid4 = tfidf_transformer.transform(bow4)
print(tfid4.toarray)
print(bow4)
print(bow4.shape)
print('predicted:',spam_detect_model.predict(tfid4)[0])
print('expected:',messages.label[3])

<bound method _cs_matrix.toarray of <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (1, 11422)>>
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7 stored elements and shape (1, 11422)>
  Coords	Values
  (0, 4066)	2
  (0, 4627)	1
  (0, 5258)	1
  (0, 6201)	1
  (0, 6219)	1
  (0, 7183)	1
  (0, 9551)	2
(1, 11422)
predicted: ham
expected: ham


In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
# fix random seed for reproducibility
tf.random.set_seed(7)


In [2]:
top_words = 500
(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=top_words)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 19s 1us/step


In [8]:
max_review_length = 500
x_train = sequence.pad_sequences(x_train,maxlen=max_review_length)
x_test= sequence.pad_sequences(x_test,maxlen=max_review_length)

In [9]:
embedding_vector_length = 32
model = Sequential()
model.add (Embedding(top_words, embedding_vector_length, input_shape=(max_review_length, )))
model.add (SimpleRNN(100))
model.add (Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print (model.summary())
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=64)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 32)        │        16,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 100)            │        13,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,401 (114.85 KB)

 Trainable params: 29,401 (114.85 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 28s 70ms/step - accuracy: 0.5493 - loss: 0.6827 - val_accuracy: 0.5830 - val_loss: 0.6617
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 26s 67ms/step - accuracy: 0.6770 - loss: 0.5949 - val_accuracy: 0.7311 - val_loss: 0.5395
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 27s 69ms/step - accuracy: 0.7377 - loss: 0.5346 - val_accuracy: 0.7466 - val_loss: 0.5165


In [10]:
scores = model.evaluate(x_test, y_test, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 74.66%
